Import Library

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Concatenate, Dropout, BatchNormalization, Layer
from tensorflow.keras.models import Model
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

Load Dataset

In [ ]:
# Upload file
# Pastikan Anda suda upload data pada files, dan sesuaikan pathnya (Cth: dataset/dataset_bersih.csv)
df = pd.read_csv('new_dataset_bersih.csv')
print(df.head())

  Python;Java;C++;JavaScript;C#;PHP;Ruby;Swift;Go;Rust;Others;Software_Development_Experience;Database_Management;Networking_Skills;Web_Development_Experience;Communication_Skills;Problem_Solving_Abilities;Teamwork_Collaboration;Time_Management;Adaptability;Career_Goals
0  9;4;9;7;8;1;5;8;6;4;7;2;3;7;5;7;5;5;9;1;Data S...                                                                                                                                                                                                                          
1  7;7;4;8;8;9;5;2;8;0;3;8;9;2;1;3;4;5;6;7;Qualit...                                                                                                                                                                                                                          
2  1;1;5;8;5;8;0;9;5;9;3;5;0;3;6;0;9;2;3;1;Comput...                                                                                                                                       

Mengelompokan label sesuai kesamaan karakterisiknya

In [ ]:
daftar_karier = {
    # Software Development
    'Front End Developer': 'Software Development',
    'Back End Developer': 'Software Development',
    'Full Stack Developer': 'Software Development',
    'Software Engineer': 'Software Development',
    'Software Developer': 'Software Development',
    'Web Developer': 'Software Development',
    'Mobile App Developer': 'Software Development',
    'Game Developer': 'Software Development',
    'Computer Programmer': 'Software Development',
    'Blockchain Developer': 'Software Development',

    # Data dan AI
    'Data Scientist': 'Data & AI',
    'Machine Learning Engineer': 'Data & AI',
    'Artificial Intelligence Engineer': 'Data & AI',
    'Business Intelligence Analyst': 'Data & AI',
    'Computer and Information Research Scientist': 'Data & AI',
    'Research Scientist': 'Data & AI',

    # Infrastrukture dan Security
    'Network Engineer': 'Infrastructure & Security',
    'Network Administrator': 'Infrastructure & Security',
    'Computer Network Architect': 'Infrastructure & Security',
    'Cloud Architect': 'Infrastructure & Security',
    'DevOps Engineer': 'Infrastructure & Security',
    'Cybersecurity Analyst': 'Infrastructure & Security',
    'Database Administrator': 'Infrastructure & Security',

    # Management dan Analysis
    'IT Project Manager': 'Management & Analysis',
    'Computer Systems Manager': 'Management & Analysis',
    'Computer and Information Systems Manager': 'Management & Analysis',
    'Systems Analyst': 'Management & Analysis',
    'Computer Systems Analyst': 'Management & Analysis',
    'IT Consultant': 'Management & Analysis',

    # Support dan Design
    'Technical Support Engineer': 'Support & Design',
    'IT Support Specialist': 'Support & Design',
    'Quality Assurance Engineer': 'Support & Design',
    'Software Tester': 'Support & Design',
    'Computer Hardware Engineer': 'Support & Design',
    'Embedded Systems Engineer': 'Support & Design',
    'UI/UX Designer': 'Support & Design',
    'Technical Writer': 'Support & Design',
    'Digital Marketing Specialist': 'Support & Design',
    'IT Sales Professional': 'Support & Design',
    'Computer Science Teacher': 'Support & Design'
}

# Memisahkan setiap kolom
if len(df.columns) == 1 and ';' in df.columns[0]:
    new_column_names = df.columns[0].split(';') # Memisahkan nama kolom
    df = df[df.columns[0]].str.split(';', expand=True) # Memisahkan isi kolom
    df.columns = new_column_names # Mengganti nama kolom hasil split

df['Target_Karier_Grouped'] = df['Career_Goals'].map(daftar_karier)

In [ ]:
# Mendefisikan hard skill dan soft skill
hard_skill = [
    "Python",
    "Java",
    "C++",
    "JavaScript",
    "C#",
    "PHP",
    "Ruby",
    "Swift",
    "Go",
    "Rust",
    # "Others", testing others tidak digunakan
    "Software_Development_Experience",
    "Database_Management",
    "Networking_Skills",
    "Web_Development_Experience"
]

soft_skill = [
    "Communication_Skills",
    "Problem_Solving_Abilities",
    "Teamwork_Collaboration",
    "Time_Management",
    "Adaptability",
]

kolom_target = list(set(daftar_karier.values()))


# Memisahkan fitur
hard_skill_data = df[hard_skill].values
soft_skill_data = df[soft_skill].values
career_goals_data = kolom_target

# Cek 5 data teratas hard, soft skill, dan carier
print("Hard Skill:")
print(hard_skill_data[:5])
print("\nSoft Skill:")
print(soft_skill_data[:5])
print("\nTarget Career:")
print(career_goals_data[:5])

Hard Skill:
[['9' '4' '9' '7' '8' '1' '5' '8' '6' '4' '2' '3' '7' '5']
 ['7' '7' '4' '8' '8' '9' '5' '2' '8' '0' '8' '9' '2' '1']
 ['1' '1' '5' '8' '5' '8' '0' '9' '5' '9' '5' '0' '3' '6']
 ['3' '7' '5' '7' '8' '7' '4' '0' '3' '2' '4' '4' '5' '7']
 ['6' '9' '2' '8' '4' '4' '6' '3' '9' '0' '6' '2' '9' '7']]

Soft Skill:
[['7' '5' '5' '9' '1']
 ['3' '4' '5' '6' '7']
 ['0' '9' '2' '3' '1']
 ['1' '7' '9' '4' '1']
 ['2' '6' '1' '3' '3']]

Target Career:
['Data & AI', 'Infrastructure & Security', 'Management & Analysis', 'Support & Design', 'Software Development']


Encode Label (Target Carier)

In [ ]:
import joblib
from tensorflow.keras.utils import to_categorical

# le = LabelEncoder()
# y_encoded = le.fit_transform(df[kolom_target])
# num_classes_career = len(np.unique(y_encoded))

le = LabelEncoder()
y_encoded = le.fit_transform(df['Target_Karier_Grouped'])
jumlah_kelas_karier = len(np.unique(y_encoded)) # Mengambil/menghitung jumlah label
y_onehot = to_categorical(y_encoded, num_classes=jumlah_kelas_karier)


# Split 1: Memsisahkan data training (80%) dan tata temporary (20% untuk val & test)
X_hard_train, X_hard_temp, X_soft_train, X_soft_temp, y_train, y_temp = train_test_split(
    hard_skill_data, soft_skill_data, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded # Menjaga proporsi kelas karier tetap seimbang
)


# Split 2: Membagi data temporary menjadi data validation (50%) dan data test (50%)
X_hard_val, X_hard_test, X_soft_val, X_soft_test, y_val, y_test = train_test_split(
    X_hard_temp, X_soft_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp # Menjaga proporsi kelas karier tetap seimbang
)


# Normalisasi data
scaler_hard = StandardScaler()
scaler_soft = StandardScaler()

# Melakukan fit & tranfrom pada data train
X_hard_train = scaler_hard.fit_transform(X_hard_train)
X_soft_train = scaler_soft.fit_transform(X_soft_train)

X_hard_val = scaler_hard.transform(X_hard_val)
X_soft_val = scaler_soft.transform(X_soft_val)

X_hard_test = scaler_hard.transform(X_hard_test)
X_soft_test = scaler_soft.transform(X_soft_test)

# Cek hasil pembagian data
print(f"Data siap!\nJumlah kelas target karier: {jumlah_kelas_karier}")
print(f"Bentuk data latih (Train) : {X_hard_train.shape[0]} baris")
print(f"Bentuk data validasi (Val): {X_hard_val.shape[0]} baris")
print(f"Bentuk data uji (Test)    : {X_hard_test.shape[0]} baris")


# Cek jumlah label
print(f"Jumlah label: {jumlah_kelas_karier}")


# Testing mengambil label/target
label_asli = le.classes_
print(label_asli)

Data siap!
Jumlah kelas target karier: 5
Bentuk data latih (Train) : 6400 baris
Bentuk data validasi (Val): 800 baris
Bentuk data uji (Test)    : 800 baris
Jumlah label: 5
['Data & AI' 'Infrastructure & Security' 'Management & Analysis'
 'Software Development' 'Support & Design']


Custom Layer

In [ ]:
class FeatureEmphasisLayer(Layer):
    # Memberikan bobot dinamis pada fitur gabungan sebelum prediksi akhir
    def __init__(self, **kwargs):
        super(FeatureEmphasisLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1],), initializer="ones", trainable=True)
        super(FeatureEmphasisLayer, self).build(input_shape)

    def call(self, inputs):
        return inputs * self.w

Define branch 1 (Hard Skill)

In [ ]:
input_hard = Input(shape=(len(hard_skill),), name="input_hard_skill")
x1 = Dense(612, activation='relu')(input_hard)
x1 = BatchNormalization()(x1)
x1 = Dropout(0.3)(x1)
x1 = Dense(512, activation='relu')(x1)
x1 = Dropout(0.2)(x1)

Define Branch 2 (Soft Skill)

In [ ]:
input_soft = Input(shape=(len(soft_skill),), name="input_soft_skill")
x2 = Dense(218, activation='relu')(input_soft)
x2 = Dense(64, activation='relu')(x2)
x2 = Dropout(0.2)(x2)

Combine brach 1 & 2

In [ ]:
combined = Concatenate()([x1, x2])

# Penerapan Custom Layer
custom_combined = FeatureEmphasisLayer(name="custom_emphasis")(combined)

x3 = Dense(562, activation='relu')(combined)
x3 = BatchNormalization()(x3)
x3 = Dropout(0.3)(x3)

x3 = Dense(282, activation='relu')(x3)

output_layer = Dense(jumlah_kelas_karier, activation='softmax', name="Prediksi_Karier")(x3)

model = Model(inputs=[input_hard, input_soft], outputs=output_layer)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_hard_skill    │ (None, 14)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 612)       │      9,180 │ input_hard_skill… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 612)       │      2,448 │ dense[0][0]       │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_soft_skill    │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 612)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 218)       │      1,308 │ input_soft_skill… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │    313,856 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │     14,016 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 64)        │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 576)       │          0 │ dropout_1[0][0],  │
│ (Concatenate)       │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 562)       │    324,274 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 562)       │      2,248 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 562)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 282)       │    158,766 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Prediksi_Karier     │ (None, 5)         │      1,415 │ dense_5[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 827,511 (3.16 MB)

 Trainable params: 825,163 (3.15 MB)

 Non-trainable params: 2,348 (9.17 KB)

Custom Callback

In [ ]:
import numpy as np
import tensorflow as tf

# Merubah tipe data
X_hard_train = X_hard_train.astype(np.float32)
X_soft_train = X_soft_train.astype(np.float32)

X_hard_val = X_hard_val.astype(np.float32)
X_soft_val = X_soft_val.astype(np.float32)

# Menghentikan training jika val_loss tidak membaik
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# Menurunkan learning rate jika model stagnan
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

# Menyimpan model terbaik
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

# Training model
history = model.fit(
    x=[X_hard_train, X_soft_train],
    y=y_train,

    validation_data=(
        [X_hard_val, X_soft_val],
        y_val
    ),

    epochs=100,
    batch_size=32,

    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ]
)

Epoch 1/100
191/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2232 - loss: 2.0497
Epoch 1: val_accuracy improved from None to 0.20875, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.2237 - loss: 1.9472 - val_accuracy: 0.2087 - val_loss: 1.6319 - learning_rate: 0.0010
Epoch 2/100
192/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2460 - loss: 1.7591
Epoch 2: val_accuracy improved from 0.20875 to 0.22875, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2439 - loss: 1.7420 - val_accuracy: 0.2288 - val_loss: 1.6502 - learning_rate: 0.0010
Epoch 3/100
197/200 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2562 - loss: 1.6632
Epoch 3: val_accuracy improved from 0.22875 to 0.23250, saving model to best_model.keras

Epoch 3: finished saving model to best_model.keras
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/

In [ ]:
import joblib

# Menyimpan model
model.save("matchstep_ai_model.keras")

# Menyimpan scaler dan encoder agar skala data ujian sama dengan data latih
joblib.dump(scaler_hard, "scaler_hard.pkl")
joblib.dump(scaler_soft, "scaler_soft.pkl")
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [ ]:
import numpy as np
import tensorflow as tf
import joblib
from tensorflow.keras.layers import Layer

# Mendeklarasi custom komponen
class FeatureEmphasisLayer(Layer):
    def __init__(self, **kwargs):
        super(FeatureEmphasisLayer, self).__init__(**kwargs)
    def build(self, input_shape):
        self.w = self.add_weight(shape=(input_shape[-1],), initializer="ones", trainable=True)
        super(FeatureEmphasisLayer, self).build(input_shape)
    def call(self, inputs):
        return inputs * self.w

class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, name='categorical_focal_loss', **kwargs):
        super(CategoricalFocalLoss, self).__init__(name=name, **kwargs)
        self.gamma = gamma
    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, tf.keras.backend.epsilon(), 1.0 - tf.keras.backend.epsilon())
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = tf.math.pow(1.0 - y_pred, self.gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=-1)

# Load model & dependensi
loaded_model = tf.keras.models.load_model(
    "matchstep_ai_model.keras",
    custom_objects={
        "FeatureEmphasisLayer": FeatureEmphasisLayer,
        "CategoricalFocalLoss": CategoricalFocalLoss
    }
)

scaler_hard = joblib.load("scaler_hard.pkl")
scaler_soft = joblib.load("scaler_soft.pkl")
label_encoder = joblib.load("label_encoder.pkl")

# Fungsi prediksi
def dapatkan_rekomendasi_detail(nilai_hard_skill, nilai_soft_skill):
    input_hard = np.array([nilai_hard_skill])
    input_soft = np.array([nilai_soft_skill])

    # Normalisasi data
    hard_scaled = scaler_hard.transform(input_hard)
    soft_scaled = scaler_soft.transform(input_soft)

    # Untuk melakukan Prediksi
    prediksi_probabilitas = loaded_model.predict([hard_scaled, soft_scaled], verbose=0)[0]

    # Untuk mendapatkan daftar semua nama karier dari labelEncoder
    daftar_karier = label_encoder.classes_

    # Menggabungkan nama karier dan probabilitasnya menggunakan zip
    matriks_skor = list(zip(daftar_karier, prediksi_probabilitas))

    # Mengurutkan matriks dari persentase probabilitas tertinggi ke terendah
    matriks_skor_terurut = sorted(matriks_skor, key=lambda x: x[1], reverse=True)

    # Mengambil rekomendasi top 1
    karier_top1 = matriks_skor_terurut[0][0]
    keyakinan_top1 = matriks_skor_terurut[0][1] * 100

    return karier_top1, keyakinan_top1, matriks_skor_terurut

# Data dumy, testing input user
kolom_hard_skill = [
    "Python", "Java", "C++", "JavaScript", "C#", "PHP", "Ruby",
    "Swift", "Go", "Rust", "Others", "Software_Development_Experience",
    "Database_Management", "Networking_Skills", "Web_Development_Experience"
]

kolom_soft_skill = [
    "Communication_Skills", "Problem_Solving_Abilities",
    "Teamwork_Collaboration", "Time_Management", "Adaptability"
]

print("\n" + "="*60)
print("SISTEM REKOMENDASI KARIER MATCHSTEP AI")
print("="*60)

# Dumy input user dengan rentang 0-9 sesuai dataset pelatihan
input_mahasiswa_hard = [7, 6, 5, 3, 0, 0, 4, 0, 4, 0, 4, 3, 5, 4]
input_mahasiswa_soft = [7, 5, 8, 0, 0]

# Menampilkan profil user
print("\n[PROFIL KEMAMPUAN MAHASISWA]")
print(f"{'Kategori / Fitur':<35} | {'Skor Input'}")
print("-" * 55)

print("A. Hard Skill & Experience:")
for nama_mk, skor in zip(kolom_hard_skill, input_mahasiswa_hard):
    print(f"   - {nama_mk:<30} | {skor:>5.1f}")

print("\nB. Soft Skill (Interpersonal):")
for nama_sk, skor in zip(kolom_soft_skill, input_mahasiswa_soft):
    print(f"   - {nama_sk:<30} | {skor:>5.1f}")
print("-" * 55)

# Proses prediksi dan menampilkan hasilnya
karier_utama, keyakinan_utama, matriks_detail = dapatkan_rekomendasi_detail(
    input_mahasiswa_hard, input_mahasiswa_soft
)

print(f"\n[HASIL ANALISIS UTAMA]")
print(f"Rekomendasi Utama : **{karier_utama}**")
print(f"Tingkat Keyakinan : {keyakinan_utama:.2f}%\n")

print("[MATRIKS KECOCOKAN KARIER (TOP 5)]")
print(f"{'Nama Karier':<35} | {'Skor Kecocokan'}")
print("-" * 55)

# Tampilkan hanya Top 5 karier teratas
for karier, prob in matriks_detail[:5]:
    persentase = prob * 100
    print(f"{karier:<35} | {persentase:>6.2f} %")

print("="*60)


SISTEM REKOMENDASI KARIER MATCHSTEP AI

[PROFIL KEMAMPUAN MAHASISWA]
Kategori / Fitur                    | Skor Input
-------------------------------------------------------
A. Hard Skill & Experience:
   - Python                         |   7.0
   - Java                           |   6.0
   - C++                            |   5.0
   - JavaScript                     |   3.0
   - C#                             |   0.0
   - PHP                            |   0.0
   - Ruby                           |   4.0
   - Swift                          |   0.0
   - Go                             |   4.0
   - Rust                           |   0.0
   - Others                         |   4.0
   - Software_Development_Experience |   3.0
   - Database_Management            |   5.0
   - Networking_Skills              |   4.0

B. Soft Skill (Interpersonal):
   - Communication_Skills           |   7.0
   - Problem_Solving_Abilities      |   5.0
   - Teamwork_Collaboration         |   8.0
   - Time_Manage